In [ ]:
# 1. Downloading the Data
import requests

URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

request = requests.get(URL)
with open("data.tar", "wb") as f:
  f.write(request.content)
print("Done")

Done


In [ ]:
import tarfile as t

with t.open('data.tar', 'r:gz') as f:
  f.extractall()

print('data.tar Is extracted , DONE :)')

/tmp/ipykernel_1530/2067672902.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  f.extractall()


data.tar Is extracted , DONE :)


In [ ]:
# 1. preparing the Data
!pip install pyprind
import pyprind
import pandas as pd
import os
import sys

print("Instalation and importing is DONE")

Instalation and importing is DONE


In [ ]:
from tqdm.auto import tqdm   ## this is the progress bar
BASEPATH = '/content/aclImdb'
labels = {'pos': 1,
          'neg': 0}
data = []

for s in ('test', 'train'):
    for l in ('pos', 'neg'):
        path = os.path.join(BASEPATH, s, l)

        for file in tqdm(sorted(os.listdir(path)), desc=f"{s}-{l}"):
            with open(os.path.join(path, file), 'r', encoding='utf-8') as infile:
                txt = infile.read()

            data.append([txt, labels[l]])

df = pd.DataFrame(data, columns=['review', 'sentiment'])


test-pos:   0%|          | 0/12500 [00:00<?, ?it/s]

test-neg:   0%|          | 0/12500 [00:00<?, ?it/s]

train-pos:   0%|          | 0/12500 [00:00<?, ?it/s]

train-neg:   0%|          | 0/12500 [00:00<?, ?it/s]

In [ ]:
df.head(1)

,review,sentiment
0,I went and saw this movie last night after bei...,1


In [ ]:
import numpy as np
np.random.seed(0)
df = df.reindex(np.random.permutation(df.index))
df.to_csv('movie_data.csv', index=False, encoding='utf-8')

### Creating the Functions

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop = stopwords.words('english')

import re

def tokenizer(text):
    """Creating the function to clean the text"""
    text = re.sub(r'<[^>]*>', '', text)

    emoticons = re.findall(
        r'(?::|;|=)(?:-)?(?:\)|\(|D|P)',
        text
    )

    text = (
        re.sub(r'[\W]+', ' ', text.lower())
        + ' '.join(emoticons).replace('-', '')
    )

    tokenized = [w for w in text.split() if w not in stop]

    return tokenized

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# checking the function
print(tokenizer('Hey i am a sushant and my goal is to become the great Philosopher'))
print("\n              WORKING !!")

['hey', 'sushant', 'goal', 'become', 'great', 'philosopher']

              WORKING !!


In [ ]:
idx = df.sample(1).index[0]  #  1. To Randomly select the index for data inspection.
print(df.loc[45887 , 'review'][-100:])
print(idx) #45887

ankenfish if you enjoy this sort of hilarious horror.<br /><br />(WHAT THE HELL WERE THEY SMOKING!?'
35356


# **Creating a mini batch**

So we are creating a batch for the data , to save from the SGD algorithm
creating the `stream_docs` function to return a one document ata a time

#### **What is the use of the `stream_doc()` function**

To read and use a file piece by piece instead of loading the whole file at once

* Without  --> bring all data at once
* with -->  one page at a time  


In [ ]:
# 1. creating a stream doc function
def stream_docs(path):
  with open(path, 'r', encoding='utf-8') as csv:
    next(csv) # Manually skipping the header

    for line in csv: # python iteration through the file

      text , label = line[:-3] , int(line[-2]) # what actually doing here is "theline is ,1\n" and line -3 remove 1\n and -2 mean 1
      yield text, label

#### **What is the use of the `mini_batch()` function**

to take a large sum of the data and collect a small chunk(batch) of records at a time.

basically with out passing the data and calculating the loss on the single data , it applied to the batches

#### **Function of both the functions**

stream_dics() --> 1 review at a time
mini_batch() --> 1000 reviews at a time (or size is choose able )

In [ ]:
# creating the Mini batch

def mini_batch(doc_stream , size):
  docs , y =[],[]

  try:
    for _ in range (size):
      text , label = next(doc_stream)
      docs.append(text)
      y.append(label)
  except StopIteration:
    return None , None
  return docs , y

### **OUT OF CORE learning**

a machine learning technique for training models on datasets too massive to fit into your computer's RAM (core memory)

And for this we can't use the `CountVectorizer` because it need the holding of the complete vocabulary in memory.

`TfidfVectorizer` Also need to keep all the features vectors of the training dataset.

`HashingVectorizer` this is needed for the ***Out of core*** Ml model trainig ..

# **Typical flow of how the Model will Work**

stream_docs()

      ↓
mini_batch(..., size=1000)

      ↓
Vectorize the 1000 reviews

      ↓
SGDClassifier.partial_fit(...)

      ↓
Get next batch of 1000 reviews

      ↓
partial_fit(...) again # `partial_fit` train data in batches

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer # 1. For Out of core learning
from sklearn.linear_model import SGDClassifier # 2. To implement the batch Learning
from tqdm import tqdm
# 3. instancing the vectorizer

vect = HashingVectorizer(decode_error= 'ignore',
                         n_features=2**21,
                         preprocessor=None,
                         tokenizer=tokenizer)  # tokenizer that is created above

clf = SGDClassifier(loss = 'log_loss',
                    random_state = 1)


# 4. creating the doc_stream
doc_stream = stream_docs(path='movie_data.csv')

# 5. Creating the classes
classes = np.array([0,1])

for _ in tqdm(range(45)):
  X_train , y_train = mini_batch(doc_stream , size=1000)
  if not X_train:
    break
  X_train = vect.transform(X_train)

  clf.partial_fit(X_train , y_train , classes = classes)


100%|██████████| 45/45 [00:48<00:00,  1.07s/it]


In [ ]:
X_test , y_test = mini_batch(doc_stream , size=5000)
X_test = vect.transform(X_test)

print(f"Accuracy : {clf.score(X_test , y_test)}")
#

Accuracy : 0.8682


In [ ]:
import pickle


# 1. saving the model

with open('Sentiment_model_minibatch.pkl' , 'wb') as f:
  pickle.dump(clf , f)
